# Proyecto de Analítica de Datos: Coffee Shop Sales
**Metodología:** CRISP-DM (Fase 1: Entendimiento del Negocio y Fase 2: Entendimiento de los Datos)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K3vsG/Avance-2-de-evidencia-1---IngDatosMasivos---Kevin-G/edit/main/Analisis_Descriptivo_Coffee_Shop_SalesAv2Ev1.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/)

> *Nota: Haz clic en las insignias de arriba para ejecutar este notebook de manera interactiva sin instalar nada en tu computadora.*

---

## 1. Entendimiento del Negocio (Business Understanding)

### 1.1 Contexto de la Empresa
La cadena de cafeterías opera en ubicaciones clave con un flujo constante de clientes diarios. Para mantener la competitividad y maximizar la rentabilidad, la administración requiere comprender los patrones de compra, el desempeño por sucursal, la demanda por categoría de producto y las horas de mayor afluencia.

### 1.2 Objetivos del Proyecto
1. **Analizar el Desempeño Operativo y Comercial:** Identificar el volumen de ventas e ingresos generados por sucursal (`store_location`) y detectar variaciones temporales.
2. **Optimización de Inventario y Menú:** Determinar los productos y categorías de mayor y menor rotación para ajustar compras de insumos y promociones.
3. **Identificación de Patrones de Consumo:** Analizar la distribución de transacciones a lo largo del día y de la semana para optimizar la programación del personal en barra.
4. **Estructuración y Calidad de Datos:** Formalizar un diccionario de datos y un modelo relacional normalizado que permita escalar la infraestructura analítica del negocio.

### 1.3 Criterios de Éxito del Negocio
- Generación de insights accionables para la toma de decisiones comerciales e inventario.
- Identificación clara de horas pico y productos estrella por ubicación.

## 2. Entendimiento de los Datos (Data Understanding)

### 2.1 Descripción de los Datos Disponibles
El conjunto de datos proviene del archivo de registro transaccional `Coffee Shop Sales.xlsx`. Contiene **149,116 registros** correspondientes al primer semestre del año 2023 (del 1 de enero al 30 de junio de 2023). Cada registro representa una línea de detalle dentro de una transacción efectuada en alguna de las sucursales de la cadena.

### 2.2 Diccionario de Datos

| Campo | Tipo de Dato | Nulo | Descripción | Rango / Ejemplo |
| :--- | :--- | :---: | :--- | :--- |
| **`transaction_id`** | `INTEGER` | No | Identificador único de cada línea de transacción (Clave Primaria). | `1` a `149,116` |
| **`transaction_date`** | `DATE` | No | Fecha en que se realizó la venta (AAAA-MM-DD). | `2023-01-01` a `2023-06-30` |
| **`transaction_time`** | `TIME` | No | Hora exacta de la transacción (HH:MM:SS). | `07:06:11` |
| **`transaction_qty`** | `INTEGER` | No | Cantidad de unidades vendidas del producto. | Min: `1`, Max: `8` |
| **`store_id`** | `INTEGER` | No | Identificador único de la sucursal. | `3` (Astoria), `5` (Lower Manhattan), `8` (Hell's Kitchen) |
| **`store_location`** | `VARCHAR(50)` | No | Nombre o ubicación geográfica de la sucursal. | `Lower Manhattan`, `Hell's Kitchen`, `Astoria` |
| **`product_id`** | `INTEGER` | No | Identificador único del producto. | `1` a `80` |
| **`unit_price`** | `DECIMAL(10,2)` | No | Precio unitario del producto en moneda local. | Min: `$0.80`, Max: `$45.00` |
| **`product_category`**| `VARCHAR(50)` | No | Categoría general del producto (9 categorías). | `Coffee`, `Tea`, `Drinking Chocolate`, `Bakery`, etc. |
| **`product_type`** | `VARCHAR(50)` | No | Subcategoría o variedad del producto (29 tipos). | `Gourmet brewed coffee`, `Brewed Chai tea` |
| **`product_detail`** | `VARCHAR(100)`| No | Presentación específica y tamaño del producto. | `Ethiopia Rg`, `Spicy Eye Opener Chai Lg` |

### 2.3 Arquitectura y Diagrama de Datos (Modelo Relacional)

```mermaid
erdiagram
    STORES ||--o{ TRANSACTIONS : "registra"
    PRODUCTS ||--o{ TRANSACTIONS : "es vendido en"

    STORES {
        int store_id PK
        string store_location
    }

    PRODUCTS {
        int product_id PK
        string product_category
        string product_type
        string product_detail
        decimal unit_price
    }

    TRANSACTIONS {
        int transaction_id PK
        date transaction_date
        time transaction_time
        int transaction_qty
        int store_id FK
        int product_id FK
    }
```

---
## 3. Carga de Datos y Configuración Inicial

In [ ]:
# Setup e Importación de Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de renderizado interactivo para HTML/GitHub
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Configuración de estilo gráfico estático
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

In [ ]:
# Carga de Datos en Colab / Entorno Local
'''
from google.colab import files
uploaded = files.upload()
'''

file_path = 'Coffee Shop Sales.xlsx'
df = pd.read_excel(file_path)

# Cálculo de la columna derivada total_sales ($)
df['total_sales'] = df['transaction_qty'] * df['unit_price']

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
# Inspección General
print("=== INFORMACIÓN GENERAL ===")
df.info()

print("\n=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())

print("\n=== RESUMEN ESTADÍSTICO ===")
display(df[['transaction_qty', 'unit_price', 'total_sales']].describe().T)

---
## 4. Análisis Descriptivo Operativo y Comercial

In [ ]:
# Desempeño Comercial por Sucursal (Gráfico Interactivo Plotly)
store_summary = df.groupby('store_location').agg(
    total_transactions=('transaction_id', 'count'),
    total_units_sold=('transaction_qty', 'sum'),
    total_revenue=('total_sales', 'sum'),
    avg_ticket=('total_sales', 'mean')
).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Ingresos Totales por Sucursal ($)', 'Total de Transacciones por Sucursal')
)

fig.add_trace(
    go.Bar(x=store_summary['store_location'], y=store_summary['total_revenue'], name='Ingresos ($)', marker_color='Viridis'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=store_summary['store_location'], y=store_summary['total_transactions'], name='Transacciones', marker_color='Magma'),
    row=1, col=2
)

fig.update_layout(height=450, title_text="Dashboard Interactivo: Desempeño por Sucursal", showlegend=False, template="plotly_white")
fig.show()

---
## 5. Análisis Exploratorio de Datos (EDA) - Respuestas a Preguntas de Negocio

In [ ]:
# Preparación de datos temporales
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['day_name'] = df['transaction_date'].dt.day_name()
df['hour'] = pd.to_datetime(df['transaction_time'].astype(str)).dt.hour

# Ordenar los días de la semana
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_name'] = pd.Categorical(df['day_name'], categories=days_order, ordered=True)

In [ ]:
# Visualización 1: ¿Qué productos se venden más? (Plotly Interactivo)
top_products = df.groupby('product_detail').agg(
    total_units=('transaction_qty', 'sum'),
    total_revenue=('total_sales', 'sum')
).sort_values(by='total_units', ascending=True).tail(10).reset_index()

fig_top = px.bar(
    top_products, 
    x='total_units', 
    y='product_detail', 
    orientation='h',
    title='Top 10 Productos Más Vendidos (Por Volumen de Unidades)',
    labels={'total_units': 'Unidades Vendidas', 'product_detail': 'Producto'},
    text='total_units',
    color='total_units',
    color_continuous_scale='Blues'
)

fig_top.update_layout(template="plotly_white", height=500)
fig_top.show()

In [ ]:
# Visualización 2: ¿Qué productos se venden por día y por hora? (Mapa de Calor Interactivo)
heatmap_data = df.pivot_table(
    index='day_name', 
    columns='hour', 
    values='transaction_id', 
    aggfunc='count', 
    fill_value=0
)

fig_hm = px.imshow(
    heatmap_data,
    labels=dict(x="Hora del Día (24h)", y="Día de la Semana", color="Transacciones"),
    x=heatmap_data.columns,
    y=heatmap_data.index,
    color_continuous_scale='YlOrRd',
    aspect="auto",
    title="Mapa de Calor: Intensidad de Transacciones por Día y Hora"
)

fig_hm.update_layout(template="plotly_white", height=450)
fig_hm.show()

In [ ]:
# Visualización 3: Demanda de Categorías por Franja Horaria del Día (Plotly Interactivo)
def assign_time_slot(hour):
    if 6 <= hour < 12:
        return 'Mañana (06:00 - 11:59)'
    elif 12 <= hour < 17:
        return 'Tarde (12:00 - 16:59)'
    else:
        return 'Noche (17:00 en adelante)'

df['time_slot'] = df['hour'].apply(assign_time_slot)
slot_products = df.groupby(['time_slot', 'product_category'])['transaction_qty'].sum().reset_index()

fig_slot = px.bar(
    slot_products,
    x='time_slot',
    y='transaction_qty',
    color='product_category',
    barmode='group',
    title='Venta de Categorías por Franja Horaria del Día',
    labels={'time_slot': 'Franja Horaria', 'transaction_qty': 'Unidades Vendidas', 'product_category': 'Categoría'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig_slot.update_layout(template="plotly_white", height=500)
fig_slot.show()

## 6. Conclusiones y Hallazgos Principales

1. **Productos Estrella:** Las bebidas calientes (`Coffee` y `Tea`) encabezan de forma constante el volumen de ventas por unidad, seguidas por productos complementarios de repostería (`Bakery`).
2. **Análisis por Día y Hora:** Las horas pico se concentran estrictamente entre las **7:00 AM y 10:00 AM**, con una actividad constante de lunes a viernes, lo que sugiere que el perfil principal de clientes realiza sus compras antes o durante su jornada laboral.
3. **Comportamiento por Franja:** Durante la tarde, aunque el volumen total disminuye, las categorías de bebidas frías y chocolates aumentan ligeramente en participación en comparación con el pico matutino.